# `unplug_charger` Transformer + Flow Matching

这份 notebook 是 AutoDL 自包含主训练入口。默认配置：
- 点云 + PointNet token
- DiT 风格 transformer backbone
- Flow Matching 策略
- `n_obs_steps = 3`
- 训练节奏、checkpoint、optimizer 尽量贴近 upstream transformer 配置


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

CANDIDATES = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = None
for cand in CANDIDATES:
    if (cand / 'pyproject.toml').exists() and (cand / 'scripts' / 'train.py').exists():
        PROJECT_ROOT = cand
        break
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate project root. Please open the notebook from the repo root or its notebooks/ directory.')

TRAIN_SCRIPT = PROJECT_ROOT / 'scripts' / 'train.py'
EVAL_ALL_SCRIPT = PROJECT_ROOT / 'scripts' / 'eval_all_checkpoints.py'
DEFAULT_CONFIG = PROJECT_ROOT / 'configs' / 'fm_autodl_lab.json'


In [ ]:
# User choices
CONFIG_PATH = DEFAULT_CONFIG
STRATEGY = 'fm'
RUN_NAME = 'unplug_charger_transformer_fm_obs3_dit_v1_retrain_noamp_v1'
DEVICE = None  # e.g. 'cuda', 'cpu'
RESUME_FROM_LATEST = True

POST_TRAIN_EVAL_EPISODES = 10
POST_TRAIN_EVAL_MAX_STEPS = 200
POST_TRAIN_EVAL_SEED = 1234
POST_TRAIN_EVAL_INCLUDE_SPECIAL = False

run_dir = PROJECT_ROOT / 'ckpt' / RUN_NAME
summary_path = run_dir / 'summary.json'
eval_results_json = run_dir / 'local_eval_results.json'
eval_plot_path = run_dir / 'local_eval_success_rate.png'

train_cmd = [
    sys.executable,
    '-u',
    str(TRAIN_SCRIPT),
    '--config',
    str(CONFIG_PATH),
    '--strategy',
    STRATEGY,
    '--run-name',
    RUN_NAME,
]
if DEVICE:
    train_cmd.extend(['--device', DEVICE])
train_cmd.append('--resume' if RESUME_FROM_LATEST else '--no-resume')

print('training command:')
print(' '.join(train_cmd))
print(f'run_dir -> {run_dir}')


In [ ]:
# Run training via the standalone CLI entry
subprocess.run(train_cmd, cwd=PROJECT_ROOT, check=True, text=True)

if not summary_path.exists():
    raise FileNotFoundError(f'Expected summary file at {summary_path}, but it was not generated.')

summary = json.loads(summary_path.read_text(encoding='utf-8'))
summary


## Notebook-safe RLBench workflow

- notebook 训练阶段故意关闭 `success_selection`，避免在 `ipykernel` 里启动 RLBench / CoppeliaSim。
- 训练后的成功率评估请使用下面的独立进程 cell；它会调用 `scripts/eval_all_checkpoints.py`。
- 评估缓存会写到 `cfg.ckpt_dir / 'local_eval_results.json'`，重复运行时会优先复用已有结果。
- `best_success.pt` 不会在 notebook 训练过程中生成；训练后请用汇总 cell 找到 success rate 最好的 checkpoint。


In [ ]:
# Optional: run standalone post-training success evaluation across all saved checkpoints
RUN_POST_TRAIN_EVAL = False

eval_cmd = [
    sys.executable,
    '-u',
    str(EVAL_ALL_SCRIPT),
    '--ckpt-epochs-dir',
    str(run_dir / 'epochs'),
    '--results-json',
    str(eval_results_json),
    '--plot-path',
    str(eval_plot_path),
    '--episodes',
    str(POST_TRAIN_EVAL_EPISODES),
    '--max-steps',
    str(POST_TRAIN_EVAL_MAX_STEPS),
    '--seed',
    str(POST_TRAIN_EVAL_SEED),
    '--headless',
]
if DEVICE:
    eval_cmd.extend(['--device', DEVICE])
if POST_TRAIN_EVAL_INCLUDE_SPECIAL:
    eval_cmd.append('--include-special')
else:
    eval_cmd.append('--no-include-special')

print(' '.join(eval_cmd))
if RUN_POST_TRAIN_EVAL:
    subprocess.run(eval_cmd, cwd=PROJECT_ROOT, check=True, text=True)
    print(f'saved eval cache -> {eval_results_json}')
    print(f'saved eval plot  -> {eval_plot_path}')


In [ ]:
# Summarize the best checkpoint from cached standalone evaluation results
if not eval_results_json.exists():
    raise FileNotFoundError(
        f'No cached eval results found at {eval_results_json}. Run the standalone eval cell first.'
    )

cached_results = json.loads(eval_results_json.read_text(encoding='utf-8'))
rows_by_path = {}
for row in cached_results.values():
    ckpt_path = row.get('path')
    success_rate = row.get('success_rate')
    if not ckpt_path or success_rate is None:
        continue
    rows_by_path.setdefault(ckpt_path, row)

candidate_rows = list(rows_by_path.values())
if not candidate_rows:
    raise RuntimeError('No successful checkpoint evaluations were found in the cached results.')

def _row_sort_key(row):
    epoch = row.get('epoch')
    epoch_sort = int(epoch) if epoch is not None else 10**9
    return (-float(row['success_rate']), epoch_sort, str(row.get('label') or ''))

best_row = sorted(candidate_rows, key=_row_sort_key)[0]
best_summary = {
    'best_ckpt_path': best_row['path'],
    'best_epoch': best_row.get('epoch'),
    'best_success_rate': float(best_row['success_rate']),
    'mean_steps': None if best_row.get('mean_steps') is None else float(best_row['mean_steps']),
}
best_summary
